# 03 · Timing test (run once, before the tuning)

This notebook is a short test. It answers two practical questions before the long runs start.

1. **Are the igraph features right?** Notebook 01 computes betweenness, closeness, eigenvector and k-core with igraph, a library written in C. Here the same features are computed again with networkx (pure Python, the most common reference library) on two real time steps, and compared with igraph and with the values saved in `B_features.csv`. The notebook also measures how long networkx takes and estimates its time on all 49 time steps.
2. **How long will the training take?** One short training per model on a GPU, with a *medium* and a *heavy* configuration, and with few (27) and many (192) input columns. From these times the notebook estimates the length of the tuning (step 03) and of the final runs (step 04).

**Runtime.** Choose a GPU (`Runtime → Change runtime type → T4 GPU`). Without a GPU the classical models are still measured and the neural ones are skipped.

**Input** (in `DATA_PATH` on Google Drive): `txs_features.csv`, `txs_classes.csv`, `txs_edgelist.csv`, and the three files of step 01 in `01-graph-feature-extractions/`.

**Output** (in `DATA_PATH/03-tuning/timing-test/`):
- `igraph_vs_networkx.csv`: the largest difference between the two libraries for each feature, and their run times;
- `timings.csv`: one row per measurement;
- `estimate.csv`: the estimated hours of tuning and of final runs for each model;
- `environment.json`: GPU, CPU and library versions.

**How to run it.** `Runtime → Run all`. It takes roughly 30–45 minutes. Each measurement is written to Drive as soon as it is done, and the measurements already on Drive are skipped: if Colab disconnects, run the notebook again and it continues where it stopped. A measurement that fails (for example, a GPU out of memory) is written with its error and does not stop the others. Set `FORCE_RECOMPUTE = True` to start from scratch.

The input preparation here is deliberately simple (`log1p` and z-score). It only has to give realistic input sizes and value ranges: the real preparation of the feature menus is done in step 02. The training numbers of this notebook are only used to plan the work, never as results.

## 0. Setup

In [1]:
# Install the libraries that Colab does not have by default (only if they are missing).
import importlib.util
import subprocess
import sys

for package, module in [('igraph', 'igraph'), ('torch_geometric', 'torch_geometric')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [2]:
# Connect Google Drive: every input and output of the project lives in DATA_PATH.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/UNIVERSITA/block-crypto/data')
FEATURE_DIR = DATA_PATH / '01-graph-feature-extractions'   # output of step 01
OUT_DIR = DATA_PATH / '03-tuning' / 'timing-test'           # output of this notebook
LOCAL_INPUT = Path('/content/input')                        # local copy of the input files

Mounted at /content/drive


In [3]:
# Copy the input files from Drive to the Colab disk: reading a local file is much faster.
import shutil

LOCAL_INPUT.mkdir(exist_ok=True)
for source in [DATA_PATH / 'txs_features.csv', DATA_PATH / 'txs_classes.csv', DATA_PATH / 'txs_edgelist.csv',
               FEATURE_DIR / 'A_features.csv', FEATURE_DIR / 'B_features.csv', FEATURE_DIR / 'C_features.csv']:
    assert source.exists(), f'{source} is missing: run the notebook of step 01 first'
    target = LOCAL_INPUT / source.name
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy(source, target)
    print(f'{source.name}: {target.stat().st_size / 1e6:,.0f} MB')

txs_features.csv: 695 MB
txs_classes.csv: 2 MB
txs_edgelist.csv: 4 MB
A_features.csv: 5 MB
B_features.csv: 25 MB
C_features.csv: 8 MB


In [4]:
import json
import os
import platform
import shutil
import time
from types import SimpleNamespace

import igraph as ig
import joblib
import networkx as nx
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn.functional as F
import torch_geometric
import xgboost
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from torch import nn
from torch_geometric.nn import ChebConv, GATv2Conv, GINConv, SAGEConv
from torch_geometric.utils import to_undirected
from xgboost import XGBClassifier

TRAIN_STEPS, VAL_STEPS, TEST_STEPS = range(1, 31), range(31, 41), range(41, 50)
EIG_ZERO = 1e-12   # same cut-off as notebook 01
SEED = 0

# Short runs: the time of one epoch is the mean over TIMED_EPOCHS epochs, after WARMUP_EPOCHS
# epochs that are not counted (the first epochs on a GPU are slower).
WARMUP_EPOCHS, TIMED_EPOCHS = 5, 30
# Full runs: one complete training with early stopping, to see after how many epochs it stops.
MAX_EPOCHS = 1000
PATIENCE_MLP, PATIENCE_GNN = 50, 100   # epochs without improvement before early stopping
LR, WEIGHT_DECAY = 1e-3, 1e-5   # the middle of the planned search ranges (log scale)

# Planned tuning, used only for the estimate at the end (the real plan is set in step 02).
MENU_WIDTH = {'local': 93, 'local_agg': 165, 'graph': 27, 'econ': 14, 'transparent': 41,
              'local_agg_graph': 192, 'none': 1}   # number of input columns of each menu
FEATURE_MENUS = ['local', 'local_agg', 'graph', 'econ', 'transparent', 'local_agg_graph']
PLAN = [   # (model, menus, tuning trials per menu)
    ('logreg', FEATURE_MENUS, 20),
    ('rf', FEATURE_MENUS, 20),
    ('xgb', FEATURE_MENUS, 40),
    ('mlp', FEATURE_MENUS, 40),
    ('sage', FEATURE_MENUS, 40),
    ('gatv2', FEATURE_MENUS, 40),
    ('cheb', FEATURE_MENUS, 40),
    ('cheb', ['none'], 20),
    ('gin', ['none'], 20),
]
FINAL_SEEDS = 3

# True = delete every output of this notebook on Drive and measure everything again.
FORCE_RECOMPUTE = False

if FORCE_RECOMPUTE and OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None
print('GPU:', GPU or 'none, so the neural models will be skipped (Runtime -> Change runtime type -> T4 GPU)')
print('CPU cores:', os.cpu_count())

GPU: Tesla T4
CPU cores: 2


In [5]:
def write_csv(df, path):
    """Write a CSV safely: first to a temporary file, then rename it.
    If the run stops while writing, no half-written file is left on Drive."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    df.to_csv(tmp, index=False, float_format='%.6g')
    os.replace(tmp, path)


def write_json(data, path):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(data, indent=2))
    os.replace(tmp, path)


def fmt_time(seconds):
    """Format a duration as '12.3s', '4m 05s' or '1h 07m'."""
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, seconds = divmod(int(seconds), 60)
    if minutes < 60:
        return f'{minutes}m {seconds:02d}s'
    hours, minutes = divmod(minutes, 60)
    return f'{hours}h {minutes:02d}m'


def timed(function, *args, **kwargs):
    """Run a function and return (its result, the seconds it took)."""
    start = time.perf_counter()
    result = function(*args, **kwargs)
    return result, time.perf_counter() - start


write_json({
    'gpu': GPU, 'cpu_cores': os.cpu_count(), 'python': platform.python_version(),
    'torch': torch.__version__, 'torch_geometric': torch_geometric.__version__,
    'xgboost': xgboost.__version__, 'sklearn': sklearn.__version__,
    'igraph': ig.__version__, 'networkx': nx.__version__,
}, OUT_DIR / 'environment.json')

## 1. Load the data

- **Labels:** 1 = illicit, 0 = licit, −1 = unknown.
- **Periods:** train = time steps 1–30, validation = 31–40, test = 41–49.
- **Evaluation rows:** the loss and the metrics use only the nodes that are labeled *and* have the economic columns of Elliptic++ (all but 519 labeled nodes). The unknown nodes stay in the graph.
- **Class weight** of the loss: $w_+ = \dfrac{\#\text{licit}}{\#\text{illicit}}$ on the train evaluation rows.
- **Inputs for this test:** the 27 graph features of step 01, and the 93 + 72 original features of Elliptic plus the 27 graph features (192 columns). The graph features that are never negative go through $\log(1 + x)$ (some are counts with very long tails); then every column gets a z-score $z = (x - \mu) / \sigma$, with $\mu$ and $\sigma$ computed on all the nodes of the train time steps.

In [6]:
features = pd.read_csv(LOCAL_INPUT / 'txs_features.csv')
classes = pd.read_csv(LOCAL_INPUT / 'txs_classes.csv')
edges = pd.read_csv(LOCAL_INPUT / 'txs_edgelist.csv')
graph_features = [pd.read_csv(LOCAL_INPUT / f'{family}_features.csv') for family in 'ABC']

TXID = features['txId'].to_numpy()
STEP = features['Time step'].to_numpy()
N = len(TXID)
for table in graph_features:
    assert (table['txId'].to_numpy() == TXID).all(), 'the files of step 01 are not in the order of txs_features.csv'

LF = [f'Local_feature_{i}' for i in range(1, 94)]
AF = [f'Aggregate_feature_{i}' for i in range(1, 73)]
STRUCT = [c for table in graph_features for c in table.columns if c != 'txId']
assert len(STRUCT) == 27

label = features[['txId']].merge(classes, on='txId', how='left')['class']
assert label.notna().all(), 'some transactions have no class'
Y = np.select([label == 1, label == 2], [1, 0], default=-1)   # class 3 = unknown -> -1
HAS_ECON = features['in_BTC_total'].notna().to_numpy()       # False for the 965 rows without Elliptic++ data
IN_EVAL = (Y >= 0) & HAS_ECON
IN_SPLIT = {name: np.isin(STEP, list(steps))
            for name, steps in [('train', TRAIN_STEPS), ('val', VAL_STEPS), ('test', TEST_STEPS)]}
for name, in_split in IN_SPLIT.items():
    rows = in_split & IN_EVAL
    print(f'{name:5}: {in_split.sum():,} nodes, {rows.sum():,} evaluation rows, '
          f'{Y[rows].sum():,} illicit ({Y[rows].mean():.2%})')
train_rows = IN_SPLIT['train'] & IN_EVAL
POS_WEIGHT = float((Y[train_rows] == 0).sum() / (Y[train_rows] == 1).sum())
print(f'class weight of the loss: {POS_WEIGHT:.3f}')

# Simple input preparation, only for this test (see the text above).
X = pd.concat([features[LF + AF]] + [t.drop(columns='txId') for t in graph_features], axis=1).astype(np.float64)
for c in STRUCT:
    if X[c].min() >= 0:
        X[c] = np.log1p(X[c])
mean, std = X[IN_SPLIT['train']].mean(), X[IN_SPLIT['train']].std().replace(0, 1)
X = ((X - mean) / std).astype(np.float32)
assert np.isfinite(X.to_numpy()).all()
WIDTH_COLUMNS = {27: STRUCT, 192: LF + AF + STRUCT}   # width 1 = constant input x = 1 (no features)

index_of = pd.Series(np.arange(N), index=TXID)
SRC = index_of.loc[edges['txId1']].to_numpy()
DST = index_of.loc[edges['txId2']].to_numpy()
print(f'{N:,} transactions, {len(SRC):,} edges')

train: 123,287 nodes, 26,750 evaluation rows, 2,954 illicit (11.04%)
val  : 38,316 nodes, 9,429 evaluation rows, 1,067 illicit (11.32%)
test : 42,166 nodes, 9,866 evaluation rows, 524 illicit (5.31%)
class weight of the loss: 8.056
203,769 transactions, 234,355 edges


## 2. igraph against networkx

The four features of family B that notebook 01 computes with igraph are computed again with networkx, on the undirected graph $\hat G_t$ of a time step:

| Feature | networkx | igraph (notebook 01) | Tolerance |
|---|---|---|---|
| `betw` | `betweenness_centrality(normalized=True)`, which divides by $\binom{n_t-1}{2}$ on an undirected graph | `betweenness(directed=False)` $/ \binom{n_t-1}{2}$ | $10^{-8}$ |
| `clo` | `closeness_centrality`: $\frac{n_t-1}{\sum_u \mathrm{dist}(u,v)}$ | `closeness(normalized=True)` | $10^{-8}$ |
| `core` | `core_number` | `coreness()` | exact |
| `eig` | `eigenvector_centrality_numpy`, $\lvert x \rvert / \max \lvert x \rvert$, values below $10^{-12}$ set to 0 | `eigenvector_centrality()`, same scaling | $10^{-4}$ |

Betweenness, closeness and k-core are exact algorithms, so the two libraries must give the same numbers (up to rounding). The eigenvector is found by an iterative solver: small differences are expected, and they are reported but do not count as a failure.

Two time steps are checked: the cheapest one and the median one, by the cost of exact betweenness, which grows like $n_t \cdot m_t$ ($m_t$ = number of edges). The same ratio is used to scale the time of the median time step to all 49 time steps.

In [7]:
def step_graph(t):
    """Graph of one time step with local node numbers 0 .. n_t - 1 (as in notebook 01)."""
    nodes = np.flatnonzero(STEP == t)
    local = np.full(N, -1)
    local[nodes] = np.arange(len(nodes))
    in_t = STEP[SRC] == t
    return nodes, local[SRC[in_t]], local[DST[in_t]]


def eig_scaled(values):
    eig = np.abs(np.asarray(values, dtype=float))
    eig /= eig.max()
    eig[eig < EIG_ZERO] = 0.0
    return eig


def networkx_features(n, src, dst):
    g = nx.Graph()
    g.add_nodes_from(range(n))
    g.add_edges_from(zip(src.tolist(), dst.tolist()))
    as_array = lambda d: np.array([d[i] for i in range(n)], dtype=float)
    b, tb = timed(nx.betweenness_centrality, g, normalized=True)
    c, tc = timed(nx.closeness_centrality, g)
    e, te = timed(nx.eigenvector_centrality_numpy, g)
    k, tk = timed(nx.core_number, g)
    values = {'betw': as_array(b), 'clo': as_array(c), 'core': as_array(k), 'eig': eig_scaled(as_array(e))}
    return values, {'betw': tb, 'clo': tc, 'core': tk, 'eig': te}


def igraph_features(n, src, dst):
    g = ig.Graph(n=n, edges=list(zip(src.tolist(), dst.tolist())), directed=False)
    b, tb = timed(g.betweenness, directed=False)
    c, tc = timed(g.closeness, normalized=True)
    e, te = timed(g.eigenvector_centrality)
    k, tk = timed(g.coreness)
    values = {'betw': np.array(b) / ((n - 1) * (n - 2) / 2), 'clo': np.array(c),
              'core': np.array(k, dtype=float), 'eig': eig_scaled(e)}
    return values, {'betw': tb, 'clo': tc, 'core': tk, 'eig': te}


TOLERANCE = {'betw': 1e-8, 'clo': 1e-8, 'core': 0.0, 'eig': 1e-4}
EXACT = ['betw', 'clo', 'core']   # the eigenvector is only reported
WORK = {t: int((STEP == t).sum()) * int((STEP[SRC] == t).sum()) for t in range(1, 50)}   # n_t * m_t
by_work = sorted(WORK, key=WORK.get)
CHECK_STEPS = [by_work[0], by_work[len(by_work) // 2]]   # the cheapest and the median time step

CHECK_FILE = OUT_DIR / 'igraph_vs_networkx.csv'
if CHECK_FILE.exists():
    check = pd.read_csv(CHECK_FILE)
    print('Comparison already on Drive: skipped.')
else:
    saved = graph_features[1]   # B_features.csv
    rows = []
    for t in CHECK_STEPS:
        nodes, src, dst = step_graph(t)
        n = len(nodes)
        print(f'time step {t}: {n:,} nodes, {len(src):,} edges ...', flush=True)
        nx_values, nx_seconds = networkx_features(n, src, dst)
        ig_values, ig_seconds = igraph_features(n, src, dst)
        for f in TOLERANCE:
            rows.append({
                'time_step': t, 'nodes': n, 'edges': len(src), 'feature': f,
                'max_diff_networkx_igraph': np.abs(nx_values[f] - ig_values[f]).max(),
                'max_diff_networkx_file': np.abs(nx_values[f] - saved[f].to_numpy()[nodes]).max(),
                'tolerance': TOLERANCE[f],
                'networkx_seconds': nx_seconds[f], 'igraph_seconds': ig_seconds[f],
            })
        print(f'  networkx {fmt_time(sum(nx_seconds.values()))}, igraph {fmt_time(sum(ig_seconds.values()))}')
    check = pd.DataFrame(rows)
    check['ok'] = check[['max_diff_networkx_igraph', 'max_diff_networkx_file']].max(axis=1) <= check['tolerance']
    write_csv(check, CHECK_FILE)
display(check)

# Exact betweenness and closeness cost about n_t * m_t: scale the median time step to all 49.
median_step = CHECK_STEPS[1]
scale = sum(WORK.values()) / WORK[median_step]
for f in ['betw', 'clo']:
    row = check[(check['time_step'] == median_step) & (check['feature'] == f)].iloc[0]
    print(f'{f}: on time step {median_step} networkx {fmt_time(row.networkx_seconds)}, igraph {fmt_time(row.igraph_seconds)}'
          f' -> estimate for all 49 time steps: networkx ~{fmt_time(row.networkx_seconds * scale)}, '
          f'igraph ~{fmt_time(row.igraph_seconds * scale)}')

CHECKS_OK = bool(check[check['feature'].isin(EXACT)]['ok'].all())
if not check[check['feature'] == 'eig']['ok'].all():
    print('Note: the eigenvector differs by more than the tolerance (iterative solvers); see the table.')
print('igraph and networkx agree.' if CHECKS_OK else
      'MISMATCH: see the rows with ok = False. The notebook goes on with the timings and stops at the end.')

time step 27: 1,089 nodes, 1,168 edges ...
  networkx 2.9s, igraph 0.1s
time step 11: 4,296 nodes, 4,656 edges ...


/tmp/ipykernel_1159/1670083561.py:30: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  result = function(*args, **kwargs)


  networkx 57.4s, igraph 1.5s


,time_step,nodes,edges,feature,max_diff_networkx_igraph,max_diff_networkx_file,tolerance,networkx_seconds,igraph_seconds,ok
0,27,1089,1168,betw,2.775558e-17,4.803288e-11,1.000000e-08,2.281246,0.068493,True
1,27,1089,1168,clo,0.000000e+00,4.994138e-11,1.000000e-08,0.539684,0.049565,True
2,27,1089,1168,core,0.000000e+00,0.000000e+00,0.000000e+00,0.004421,0.000104,True
3,27,1089,1168,eig,2.753209e-15,4.512987e-11,1.000000e-04,0.041296,0.005080,True
4,11,4296,4656,betw,2.775558e-17,4.712786e-11,1.000000e-08,47.937108,0.876851,True
5,11,4296,4656,clo,0.000000e+00,4.967847e-11,1.000000e-08,9.443454,0.592000,True
6,11,4296,4656,core,0.000000e+00,0.000000e+00,0.000000e+00,0.014874,0.000285,True
7,11,4296,4656,eig,2.759362e-16,4.920175e-12,1.000000e-04,0.018079,0.014245,True


betw: on time step 11 networkx 47.9s, igraph 0.9s -> estimate for all 49 time steps: networkx ~45m 13s, igraph ~49.6s
clo: on time step 11 networkx 9.4s, igraph 0.6s -> estimate for all 49 time steps: networkx ~8m 54s, igraph ~33.5s
igraph and networkx agree.


## 3. Training times

Every model is trained the way the tuning will train it:
- **models without graph** (Logistic Regression, Random Forest, XGBoost, MLP) see only the evaluation rows of the train period;
- **GNNs** see the graph of the train period (time steps 1–30, unknown nodes included, edges made undirected), and are evaluated on the graph of the validation period. The loss uses only the evaluation rows;
- **neural models** (MLP and GNNs): full batch, Adam, binary cross-entropy with the class weight $w_+$, learning rate $10^{-3}$, weight decay $10^{-5}$. After each epoch they predict the validation rows and compute the PR-AUC, as early stopping will do.

Two configurations per model: **medium**, a typical trial, and **heavy**, the most expensive corner of the planned search space.

| Model | medium | heavy |
|---|---|---|
| Logistic Regression | $C = 1$ | $C = 100$ (weak regularization, more iterations) |
| Random Forest (500 trees) | `max_features='sqrt'`, `min_samples_leaf=2` | `max_features=0.5`, `min_samples_leaf=1` |
| XGBoost (up to 2,000 trees, early stopping after 100) | learning rate 0.1, depth 6 | learning rate 0.01, depth 10 |
| MLP | 2 layers × 128 | 3 layers × 256 |
| GraphSAGE | 2 layers × 128, mean, +Lin | 3 layers × 256, max, +Lin+Skip |
| GATv2 | 2 layers, 4 heads × 32, +Lin | 3 layers, 8 heads × 64, +Lin+Skip |
| ChebNet | 2 layers × 128, $K = 3$, +Lin | 3 layers × 256, $K = 4$, +Lin+Skip |
| GIN (only with $x = 1$) | 3 layers × 64 | 5 layers × 128 |

Variants (as in Ferretti, D'Angelo and Ghini, 2025): *base* = the last convolution gives the output; *+Lin* = a linear layer after the convolutions; *+Lin+Skip* = like +Lin, with the output of the first convolution added to the last one.

**What is measured:**
- classical models: the time of one `fit`; XGBoost on CPU and on GPU; for the Random Forest also the size of the saved model, to decide whether every trained forest can be kept on Drive;
- neural models, short run: the time of one epoch (training step + validation), the peak GPU memory, and the time of one extra prediction on the train rows, which is the cost of tracking overfitting during training;
- neural models, full run (medium configuration only): a complete training with early stopping, to see how many epochs a trial lasts.

In [8]:
def period_graph(split, columns):
    """The graph of one period: its nodes, the edges between them made undirected, the inputs
    (`columns = None` gives the constant input x = 1), the labels and the evaluation mask."""
    nodes = np.flatnonzero(IN_SPLIT[split])
    local = np.full(N, -1)
    local[nodes] = np.arange(len(nodes))
    keep = IN_SPLIT[split][SRC]   # no edge crosses two time steps, so both ends are in the period
    edge_index = torch.tensor(np.vstack([local[SRC[keep]], local[DST[keep]]]), dtype=torch.long)
    x = torch.ones(len(nodes), 1) if columns is None else torch.tensor(X[columns].to_numpy()[nodes])
    mask = IN_EVAL[nodes]
    return SimpleNamespace(
        x=x.to(DEVICE),
        edge_index=to_undirected(edge_index, num_nodes=len(nodes)).to(DEVICE),
        y=torch.tensor(Y[nodes] == 1, dtype=torch.float32, device=DEVICE),
        mask=torch.tensor(mask, device=DEVICE),
        y_eval=(Y[nodes][mask] == 1).astype(int))


def labeled_rows(split, columns):
    """The inputs of a model without graph: the evaluation rows of the period, no edges."""
    rows = IN_SPLIT[split] & IN_EVAL
    y = (Y[rows] == 1).astype(int)
    return SimpleNamespace(
        x=torch.tensor(X[columns].to_numpy()[rows], device=DEVICE), edge_index=None,
        y=torch.tensor(y, dtype=torch.float32, device=DEVICE),
        mask=torch.ones(len(y), dtype=torch.bool, device=DEVICE), y_eval=y)


_cache = {}
def data(kind, width, split):
    """kind = 'graph' (GNNs) or 'rows' (MLP); width = 27, 192, or 1 for the constant input."""
    key = (kind, width, split)
    if key not in _cache:
        columns = None if width == 1 else WIDTH_COLUMNS[width]
        _cache[key] = period_graph(split, columns) if kind == 'graph' else labeled_rows(split, columns)
    return _cache[key]


def table(split, width):
    """Numpy inputs and labels for the classical models."""
    rows = IN_SPLIT[split] & IN_EVAL
    return X[WIDTH_COLUMNS[width]].to_numpy()[rows], (Y[rows] == 1).astype(int)

In [9]:
class MLP(nn.Module):
    def __init__(self, d_in, hidden, layers, dropout):
        super().__init__()
        dims = [d_in] + [hidden] * layers
        self.hidden = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.out = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index=None):
        for layer in self.hidden:
            x = F.dropout(F.relu(layer(x)), self.dropout, self.training)
        return self.out(x).squeeze(-1)


def make_conv(kind, d_in, d_out, cfg):
    if kind == 'sage':
        return SAGEConv(d_in, d_out, aggr=cfg['aggr'])
    if kind == 'gatv2':
        heads = cfg['heads'] if d_out > 1 else 1
        return GATv2Conv(d_in, d_out // heads, heads=heads, dropout=cfg['att_dropout'])
    if kind == 'cheb':
        return ChebConv(d_in, d_out, K=cfg['K'], normalization='sym')
    if kind == 'gin':
        return GINConv(nn.Sequential(nn.Linear(d_in, d_out), nn.ReLU(), nn.Linear(d_out, d_out)),
                       train_eps=cfg['train_eps'])
    raise ValueError(kind)


class GNN(nn.Module):
    """`layers` graph convolutions in one of three variants: 'base' (the last convolution gives
    the output), 'lin' (a linear layer after the convolutions), 'skip' (like 'lin', and the
    output of the first convolution is added to the output of the last one)."""

    def __init__(self, kind, d_in, hidden, layers, variant, dropout, **cfg):
        super().__init__()
        dims = [d_in] + [hidden] * layers
        if variant == 'base':
            dims[-1] = 1
        self.convs = nn.ModuleList(make_conv(kind, a, b, cfg) for a, b in zip(dims[:-1], dims[1:]))
        self.lin = None if variant == 'base' else nn.Linear(hidden, 1)
        self.variant, self.dropout = variant, dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if self.lin is not None or i < len(self.convs) - 1:
                x = F.dropout(F.relu(x), self.dropout, self.training)
            if i == 0:
                first = x
        if self.variant == 'skip':
            x = x + first
        if self.lin is not None:
            x = self.lin(x)
        return x.squeeze(-1)


NEURAL = {   # model: {configuration: hyper-parameters} (see the table above)
    'mlp':   {'medium': dict(hidden=128, layers=2, dropout=0.3),
              'heavy':  dict(hidden=256, layers=3, dropout=0.3)},
    'sage':  {'medium': dict(hidden=128, layers=2, variant='lin', dropout=0.3, aggr='mean'),
              'heavy':  dict(hidden=256, layers=3, variant='skip', dropout=0.3, aggr='max')},
    'gatv2': {'medium': dict(hidden=4 * 32, layers=2, variant='lin', dropout=0.3, heads=4, att_dropout=0.3),
              'heavy':  dict(hidden=8 * 64, layers=3, variant='skip', dropout=0.3, heads=8, att_dropout=0.3)},
    'cheb':  {'medium': dict(hidden=128, layers=2, variant='lin', dropout=0.3, K=3),
              'heavy':  dict(hidden=256, layers=3, variant='skip', dropout=0.3, K=4)},
    'gin':   {'medium': dict(hidden=64, layers=3, variant='lin', dropout=0.3, train_eps=True),
              'heavy':  dict(hidden=128, layers=5, variant='lin', dropout=0.3, train_eps=True)},
}


def build(model_name, cfg, d_in):
    if model_name == 'mlp':
        return MLP(d_in, **cfg).to(DEVICE)
    return GNN(model_name, d_in, **cfg).to(DEVICE)

In [10]:
def sync():
    """Wait for the GPU to finish, so that the clock measures the real work."""
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def train(model, train_data, val_data, max_epochs, patience):
    """Full-batch training with early stopping on the validation PR-AUC.
    Returns the time of every epoch (training step + validation) and where it stopped."""
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, dtype=torch.float32, device=DEVICE))
    best_ap, best_epoch, times = -1.0, 0, []
    for epoch in range(1, max_epochs + 1):
        sync()
        start = time.perf_counter()
        model.train()
        optimizer.zero_grad()
        out = model(train_data.x, train_data.edge_index)
        loss = loss_fn(out[train_data.mask], train_data.y[train_data.mask])
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            score = model(val_data.x, val_data.edge_index)[val_data.mask]
        ap = average_precision_score(val_data.y_eval, score.float().cpu().numpy())
        times.append(time.perf_counter() - start)
        if ap > best_ap:
            best_ap, best_epoch = ap, epoch
        elif epoch - best_epoch >= patience:
            break
    return SimpleNamespace(times=times, epochs=epoch, best_epoch=best_epoch, best_ap=best_ap)


def train_eval_seconds(model, train_data, repeats=5):
    """Time of one extra prediction on the train rows plus its PR-AUC: the cost of tracking overfitting."""
    model.eval()
    sync()
    start = time.perf_counter()
    for _ in range(repeats):
        with torch.no_grad():
            score = model(train_data.x, train_data.edge_index)[train_data.mask]
        average_precision_score(train_data.y_eval, score.float().cpu().numpy())
    return (time.perf_counter() - start) / repeats

In [11]:
def time_logreg(config, width):
    X_train, y_train = table('train', width)
    model = LogisticRegression(C={'medium': 1.0, 'heavy': 100.0}[config], max_iter=5000)
    _, seconds = timed(model.fit, X_train, y_train)
    return {'model': 'logreg', 'config': config, 'width': width, 'device': 'cpu', 'fit_seconds': seconds}


RF_CONFIGS = {'medium': dict(max_features='sqrt', min_samples_leaf=2),
              'heavy': dict(max_features=0.5, min_samples_leaf=1)}

def time_rf(config, width):
    X_train, y_train = table('train', width)
    model = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED, **RF_CONFIGS[config])
    _, seconds = timed(model.fit, X_train, y_train)
    path = Path('/content/rf_size_test.joblib')
    joblib.dump(model, path)
    size = path.stat().st_size / 1e6
    joblib.dump(model, path, compress=3)
    size_compressed = path.stat().st_size / 1e6
    path.unlink()
    return {'model': 'rf', 'config': config, 'width': width, 'device': 'cpu', 'fit_seconds': seconds,
            'model_mb': size, 'model_mb_compressed': size_compressed}


XGB_CONFIGS = {'medium': dict(learning_rate=0.1, max_depth=6),
               'heavy': dict(learning_rate=0.01, max_depth=10)}

def time_xgb(config, width, device):
    X_train, y_train = table('train', width)
    X_val, y_val = table('val', width)
    model = XGBClassifier(n_estimators=2000, early_stopping_rounds=100, eval_metric='aucpr', tree_method='hist',
                          device=device, random_state=SEED, **XGB_CONFIGS[config])
    _, seconds = timed(model.fit, X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return {'model': 'xgb', 'config': config, 'width': width, 'device': device, 'fit_seconds': seconds,
            'best_iteration': model.best_iteration}


def time_neural(model_name, config, width, full_run):
    torch.manual_seed(SEED)
    kind = 'rows' if model_name == 'mlp' else 'graph'
    train_data, val_data = data(kind, width, 'train'), data(kind, width, 'val')
    model = build(model_name, NEURAL[model_name][config], train_data.x.shape[1])
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    if full_run:
        run = train(model, train_data, val_data, MAX_EPOCHS, PATIENCE_MLP if model_name == 'mlp' else PATIENCE_GNN)
    else:
        run = train(model, train_data, val_data, WARMUP_EPOCHS + TIMED_EPOCHS, patience=10**9)
    row = {
        'model': model_name, 'config': config, 'width': width, 'device': DEVICE.type,
        'seconds_per_epoch': float(np.mean(run.times[WARMUP_EPOCHS:] or run.times)),
        'train_eval_seconds': train_eval_seconds(model, train_data),
        'n_params': sum(p.numel() for p in model.parameters()),
        'peak_gpu_mb': torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == 'cuda' else None,
    }
    if full_run:
        row.update(fit_seconds=float(np.sum(run.times)), epochs=run.epochs, best_epoch=run.best_epoch,
                   best_val_ap=run.best_ap)
    return row

In [12]:
TIMINGS_FILE = OUT_DIR / 'timings.csv'

def load_timings():
    return pd.read_csv(TIMINGS_FILE) if TIMINGS_FILE.exists() else pd.DataFrame(columns=['job', 'status'])


def run_jobs(jobs):
    """Run a list of (name, function) measurements. Each result is written to Drive at once;
    the measurements already done are skipped; a failing one is written with its error."""
    done = set(load_timings().query("status == 'ok'")['job'])
    todo = [(name, function) for name, function in jobs if name not in done]
    print(f'{len(jobs) - len(todo)} measurements already on Drive, {len(todo)} to do.')
    start = time.time()
    for i, (name, function) in enumerate(todo, 1):
        job_start = time.time()
        try:
            row = {'status': 'ok', **function()}
        except Exception as error:   # one failing model must not stop the test
            row = {'status': f'error: {type(error).__name__}: {str(error)[:200]}'}
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
        row = pd.DataFrame([{'job': name, 'gpu': GPU, **row}])
        old = load_timings()
        old = old[old['job'] != name]
        write_csv(row if old.empty else pd.concat([old, row], ignore_index=True), TIMINGS_FILE)
        elapsed = time.time() - start
        print(f'[{i}/{len(todo)}] {name}: {row.at[0, "status"]} in {fmt_time(time.time() - job_start)} | '
              f'elapsed {fmt_time(elapsed)}, left ~{fmt_time(elapsed / i * (len(todo) - i))}', flush=True)

### 3a. Classical models (CPU; XGBoost also on GPU)

In [13]:
jobs = []
for width in [27, 192]:
    for config in ['medium', 'heavy']:
        jobs.append((f'logreg|{config}|{width}', lambda c=config, w=width: time_logreg(c, w)))
        jobs.append((f'rf|{config}|{width}', lambda c=config, w=width: time_rf(c, w)))
        for device in ['cpu', 'cuda'] if DEVICE.type == 'cuda' else ['cpu']:
            jobs.append((f'xgb|{config}|{width}|{device}', lambda c=config, w=width, d=device: time_xgb(c, w, d)))
run_jobs(jobs)

0 measurements already on Drive, 16 to do.
[1/16] logreg|medium|27: ok in 0.6s | elapsed 0.6s, left ~9.2s
[2/16] rf|medium|27: ok in 21.8s | elapsed 22.4s, left ~2m 36s
[3/16] xgb|medium|27|cpu: ok in 1.3s | elapsed 23.8s, left ~1m 42s
[4/16] xgb|medium|27|cuda: ok in 0.8s | elapsed 24.5s, left ~1m 13s
[5/16] logreg|heavy|27: ok in 0.5s | elapsed 25.0s, left ~55.0s
[6/16] rf|heavy|27: ok in 45.0s | elapsed 1m 09s, left ~1m 56s
[7/16] xgb|heavy|27|cpu: ok in 5.9s | elapsed 1m 15s, left ~1m 37s
[8/16] xgb|heavy|27|cuda: ok in 2.0s | elapsed 1m 17s, left ~1m 17s
[9/16] logreg|medium|192: ok in 2.1s | elapsed 1m 19s, left ~1m 02s
[10/16] rf|medium|192: ok in 46.7s | elapsed 2m 06s, left ~1m 16s
[11/16] xgb|medium|192|cpu: ok in 12.1s | elapsed 2m 18s, left ~1m 03s
[12/16] xgb|medium|192|cuda: ok in 2.3s | elapsed 2m 21s, left ~47.0s
[13/16] logreg|heavy|192: ok in 5.1s | elapsed 2m 26s, left ~33.7s
[14/16] rf|heavy|192: ok in 7m 21s | elapsed 9m 47s, left ~1m 23s
[15/16] xgb|heavy|192|cpu:

### 3b. Neural models (GPU)

In [14]:
if DEVICE.type != 'cuda':
    print('No GPU: the neural models are skipped.')
else:
    jobs = []
    for model_name in ['mlp', 'sage', 'gatv2', 'cheb']:
        for config in ['medium', 'heavy']:
            for width in [27, 192]:
                jobs.append((f'{model_name}|{config}|{width}|epochs',
                             lambda m=model_name, c=config, w=width: time_neural(m, c, w, full_run=False)))
    for model_name in ['cheb', 'gin']:   # the menu without features (x = 1)
        for config in ['medium', 'heavy']:
            jobs.append((f'{model_name}|{config}|1|epochs',
                         lambda m=model_name, c=config: time_neural(m, c, 1, full_run=False)))
    # One complete training per model, to see how many epochs early stopping needs.
    for model_name, width in [('mlp', 192), ('sage', 192), ('gatv2', 192), ('cheb', 192), ('cheb', 1), ('gin', 1)]:
        jobs.append((f'{model_name}|medium|{width}|full',
                     lambda m=model_name, w=width: time_neural(m, 'medium', w, full_run=True)))
    run_jobs(jobs)

display(load_timings())

0 measurements already on Drive, 26 to do.
[1/26] mlp|medium|27|epochs: ok in 2.4s | elapsed 2.4s, left ~1m 00s
[2/26] mlp|medium|192|epochs: ok in 1.8s | elapsed 4.2s, left ~50.3s
[3/26] mlp|heavy|27|epochs: ok in 0.6s | elapsed 4.8s, left ~36.7s
[4/26] mlp|heavy|192|epochs: ok in 0.6s | elapsed 5.3s, left ~29.4s
[5/26] sage|medium|27|epochs: ok in 1.3s | elapsed 6.7s, left ~28.1s
[6/26] sage|medium|192|epochs: ok in 2.1s | elapsed 8.8s, left ~29.2s


/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

[7/26] sage|heavy|27|epochs: ok in 6.0s | elapsed 14.8s, left ~40.2s


/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

[8/26] sage|heavy|192|epochs: ok in 6.7s | elapsed 21.5s, left ~48.4s
[9/26] gatv2|medium|27|epochs: ok in 4.7s | elapsed 26.2s, left ~49.5s
[10/26] gatv2|medium|192|epochs: ok in 4.8s | elapsed 31.0s, left ~49.6s
[11/26] gatv2|heavy|27|epochs: ok in 31.2s | elapsed 1m 02s, left ~1m 24s
[12/26] gatv2|heavy|192|epochs: ok in 32.3s | elapsed 1m 34s, left ~1m 50s
[13/26] cheb|medium|27|epochs: ok in 2.6s | elapsed 1m 37s, left ~1m 37s
[14/26] cheb|medium|192|epochs: ok in 3.7s | elapsed 1m 40s, left ~1m 26s
[15/26] cheb|heavy|27|epochs: ok in 13.7s | elapsed 1m 54s, left ~1m 23s
[16/26] cheb|heavy|192|epochs: ok in 15.8s | elapsed 2m 10s, left ~1m 21s
[17/26] cheb|medium|1|epochs: ok in 2.4s | elapsed 2m 12s, left ~1m 10s
[18/26] cheb|heavy|1|epochs: ok in 13.5s | elapsed 2m 26s, left ~1m 04s
[19/26] gin|medium|1|epochs: ok in 1.3s | elapsed 2m 27s, left ~54.3s
[20/26] gin|heavy|1|epochs: ok in 3.9s | elapsed 2m 31s, left ~45.4s
[21/26] mlp|medium|192|full: ok in 0.7s | elapsed 2m 32s, le

,job,gpu,status,model,config,width,device,fit_seconds,model_mb,model_mb_compressed,best_iteration,seconds_per_epoch,train_eval_seconds,n_params,peak_gpu_mb,epochs,best_epoch,best_val_ap
0,logreg|medium|27,Tesla T4,ok,logreg,medium,27,cpu,0.562064,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rf|medium|27,Tesla T4,ok,rf,medium,27,cpu,18.353500,111.4200,25.06800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,xgb|medium|27|cpu,Tesla T4,ok,xgb,medium,27,cpu,1.244790,NaN,NaN,53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,xgb|medium|27|cuda,Tesla T4,ok,xgb,medium,27,cuda,0.718762,NaN,NaN,53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,logreg|heavy|27,Tesla T4,ok,logreg,heavy,27,cpu,0.398975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,rf|heavy|27,Tesla T4,ok,rf,heavy,27,cpu,42.976900,114.6550,24.25620,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,xgb|heavy|27|cpu,Tesla T4,ok,xgb,heavy,27,cpu,5.801260,NaN,NaN,261.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,xgb|heavy|27|cuda,Tesla T4,ok,xgb,heavy,27,cuda,1.942940,NaN,NaN,261.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,logreg|medium|192,Tesla T4,ok,logreg,medium,192,cpu,1.888420,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,rf|medium|192,Tesla T4,ok,rf,medium,192,cpu,45.776600,24.4232,6.54991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Estimate of the tuning and of the final runs

For each model and menu, the time of one trial is:
- classical models: the time of one `fit`, interpolated linearly between 27 and 192 columns for the width of the menu (the menu with 14 columns uses the time of 27); for XGBoost the faster of CPU and GPU;
- neural models: the time of one epoch (interpolated in the same way) × the number of epochs of the full run of that model.

Then:

$$\text{tuning hours} = \sum_{\text{menus}} \text{trials} \times \text{trial time}, \qquad \text{final-run hours} = \sum_{\text{menus}} 3 \times \text{trial time}.$$

The *medium* row is a typical value, the *heavy* row an upper bound. The estimate does not count the pruning of the tuning (bad trials are stopped after a few epochs), so the real tuning should be shorter.

In [15]:
done = load_timings().query("status == 'ok'")


def at_width(values, width):
    """Linear interpolation between the widths 27 and 192 (the time of 27 below 27)."""
    if width in values:
        return values[width]
    if 27 not in values or 192 not in values:
        return np.nan
    if width < 27:
        return values[27]
    return values[27] + (values[192] - values[27]) * (width - 27) / (192 - 27)


def per_width(rows, column):
    return dict(zip(rows['width'].astype(int), rows[column]))


def trial_seconds(model_name, config, width):
    """(seconds of one trial, device)."""
    rows = done[(done['model'] == model_name) & (done['config'] == config)]
    if model_name in ('logreg', 'rf'):
        return at_width(per_width(rows, 'fit_seconds'), width), 'cpu'
    if model_name == 'xgb':
        options = {d: at_width(per_width(rows[rows['device'] == d], 'fit_seconds'), width)
                   for d in rows['device'].unique()}
        options = {d: s for d, s in options.items() if not np.isnan(s)}
        best = min(options, key=options.get)   # ValueError if no device was measured
        return options[best], best
    short = rows[rows['job'].str.endswith('|epochs')]
    full = done[(done['model'] == model_name) & done['job'].str.endswith('|full')]
    epochs = full[(full['width'] == 1) == (width == 1)]['epochs'].iloc[0]
    return at_width(per_width(short, 'seconds_per_epoch'), width) * epochs, 'cuda'


rows = []
for model_name, menus, trials in PLAN:
    for config in ['medium', 'heavy']:
        try:
            per_menu = [trial_seconds(model_name, config, MENU_WIDTH[m]) for m in menus]
            seconds, device = [s for s, _ in per_menu], per_menu[0][1]
        except (KeyError, IndexError, ValueError):   # a measurement is missing (no GPU, or an error)
            seconds, device = [np.nan] * len(menus), 'missing'
        rows.append({'model': model_name, 'menus': ', '.join(menus), 'trials_per_menu': trials,
                     'config': config, 'device': device, 'mean_trial_seconds': np.mean(seconds),
                     'tuning_hours': trials * np.sum(seconds) / 3600,
                     'final_runs_hours': FINAL_SEEDS * np.sum(seconds) / 3600})
estimate = pd.DataFrame(rows)
write_csv(estimate, OUT_DIR / 'estimate.csv')
display(estimate.round(3))
display(estimate.groupby(['config', 'device'])[['tuning_hours', 'final_runs_hours']].sum().round(2))

,model,menus,trials_per_menu,config,device,mean_trial_seconds,tuning_hours,final_runs_hours
0,logreg,"local, local_agg, graph, econ, transparent, lo...",20,medium,cpu,1.075,0.036,0.005
1,logreg,"local, local_agg, graph, econ, transparent, lo...",20,heavy,cpu,2.168,0.072,0.011
2,rf,"local, local_agg, graph, econ, transparent, lo...",20,medium,cpu,28.963,0.965,0.145
3,rf,"local, local_agg, graph, econ, transparent, lo...",20,heavy,cpu,196.679,6.556,0.983
4,xgb,"local, local_agg, graph, econ, transparent, lo...",40,medium,cuda,1.235,0.082,0.006
5,xgb,"local, local_agg, graph, econ, transparent, lo...",40,heavy,cuda,4.170,0.278,0.021
6,mlp,"local, local_agg, graph, econ, transparent, lo...",40,medium,cuda,1.441,0.096,0.007
7,mlp,"local, local_agg, graph, econ, transparent, lo...",40,heavy,cuda,1.514,0.101,0.008
8,sage,"local, local_agg, graph, econ, transparent, lo...",40,medium,cuda,5.259,0.351,0.026
9,sage,"local, local_agg, graph, econ, transparent, lo...",40,heavy,cuda,26.918,1.795,0.135


tuning_hours  final_runs_hours
config device                                
heavy  cpu             6.63              0.99
       cuda           21.03              1.71
medium cpu             1.00              0.15
       cuda            3.64              0.30

In [16]:
drive.flush_and_unmount()
assert CHECKS_OK, 'igraph and networkx disagree: see igraph_vs_networkx.csv'
print('Done. Every file is on Drive in', OUT_DIR)

Done. Every file is on Drive in /content/drive/MyDrive/UNIVERSITA/block-crypto/data/03-tuning/timing-test
